In [ ]:
import numpy as np
import pandas as pd
from math import radians, sin, cos
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Wczytanie danych godzinowych z Snowflake
# ─────────────────────────────────────────────────────────────────────────────

from snowflake.snowpark.context import get_active_session
session = get_active_session()

df = session.table("DANE_TASK3").to_pandas()
df.columns = df.columns.str.lower()
df = df.rename(columns={"deviceid": "deviceId"})

df["hour"]        = pd.to_datetime(df["hour"])
df["year"]        = df["hour"].dt.year
df["month"]       = df["hour"].dt.month
df["day"]         = df["hour"].dt.day
df["hour_of_day"] = df["hour"].dt.hour
df["day_of_week"] = df["hour"].dt.dayofweek
df["day_of_year"] = df["hour"].dt.dayofyear

print(f"Łącznie: {len(df):,} wierszy | {df['deviceId'].nunique()} urządzeń")
print(f"Zakres:  {df['hour'].min()} → {df['hour'].max()}")
print(f"Kolumny: {df.columns.tolist()}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Podział train / val / test (na poziomie GODZINOWYM)
# ─────────────────────────────────────────────────────────────────────────────

train_mask = (
    ((df["year"] == 2024) & (df["month"] >= 10)) |
    ((df["year"] == 2025) & (df["month"] <= 4))
)
val_mask  = (df["year"] == 2025) & (df["month"].isin([5, 6]))
test_mask = (df["year"] == 2025) & (df["month"].isin([7, 8, 9, 10]))

df_tr = df[train_mask].copy().reset_index(drop=True)
df_vl = df[val_mask].copy().reset_index(drop=True)
df_ts = df[test_mask].copy().reset_index(drop=True)

print(f"Train : {len(df_tr):,}  | Oct 2024 – Apr 2025")
print(f"Val   : {len(df_vl):,}  | May – Jun 2025")
print(f"Test  : {len(df_ts):,}  | Jul – Oct 2025")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Feature engineering (na poziomie godzinowym)
# ─────────────────────────────────────────────────────────────────────────────

TEMP_COLS = [f"t{i}_mean" for i in range(1, 14)]
TEMP_COLS = [c for c in TEMP_COLS if c in df.columns]

DSO_CENTROIDS = {
    "Enea":   (52.41, 16.93),
    "Energa": (54.35, 18.64),
    "PGE":    (52.22, 21.01),
    "Tauron": (50.06, 19.94),
}

DSO_CLIMATE_NORMS = {
    "Enea":   {1:-1.5,2:-0.5,3:3.5,4:9.0,5:14.5,6:17.5,7:19.5,8:19.0,9:14.0,10:8.5,11:3.0,12:-0.5},
    "Energa": {1:-1.0,2:-0.5,3:2.5,4:8.0,5:13.0,6:16.5,7:18.5,8:18.0,9:13.5,10:8.0,11:3.0,12:0.0},
    "PGE":    {1:-2.5,2:-1.5,3:3.0,4:9.5,5:15.0,6:18.0,7:20.0,8:19.5,9:14.0,10:8.5,11:2.5,12:-1.5},
    "Tauron": {1:-3.0,2:-2.0,3:2.5,4:8.5,5:13.5,6:16.5,7:18.5,8:18.0,9:13.0,10:7.5,11:2.0,12:-2.0},
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def assign_dso(lat, lon):
    return min(DSO_CENTROIDS, key=lambda n: haversine_km(lat, lon, *DSO_CENTROIDS[n]))


def add_features_hourly(df: pd.DataFrame, df_train_ref: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Cechy cykliczne czasu
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24).astype(np.float32)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24).astype(np.float32)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12).astype(np.float32)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12).astype(np.float32)
    df["dow_sin"]   = np.sin(2 * np.pi * df["day_of_week"] / 7).astype(np.float32)
    df["dow_cos"]   = np.cos(2 * np.pi * df["day_of_week"] / 7).astype(np.float32)
    df["doy_sin"]   = np.sin(2 * np.pi * df["day_of_year"] / 365).astype(np.float32)
    df["doy_cos"]   = np.cos(2 * np.pi * df["day_of_year"] / 365).astype(np.float32)

    # Czy godzina szczytu (6-9, 17-21)
    df["is_peak_hour"] = df["hour_of_day"].isin([6,7,8,9,17,18,19,20,21]).astype(np.int8)
    df["is_night"]     = df["hour_of_day"].isin([0,1,2,3,4,5]).astype(np.int8)

    # 2. Delty temperatur (fizyczne — stabilne między sezonami!)
    if "t5_mean" in df.columns and "t4_mean" in df.columns:
        df["load_source_delta"] = (df["t5_mean"] - df["t4_mean"]).astype(np.float32)
    if "t6_mean" in df.columns and "t5_mean" in df.columns:
        df["load_hex_delta"]    = (df["t6_mean"] - df["t5_mean"]).astype(np.float32)
    if "t2_mean" in df.columns and "t1_mean" in df.columns:
        df["indoor_outdoor"]    = (df["t2_mean"] - df["t1_mean"]).astype(np.float32)
    if "t10_mean" in df.columns and "t11_mean" in df.columns:
        df["air_hex_delta"]     = (df["t10_mean"] - df["t11_mean"]).astype(np.float32)
    if "t8_mean" in df.columns and "t9_mean" in df.columns:
        df["cooling_ch_delta"]  = (df["t8_mean"] - df["t9_mean"]).astype(np.float32)
    if "t12_mean" in df.columns and "t13_mean" in df.columns:
        df["load_hex2_delta"]   = (df["t13_mean"] - df["t12_mean"]).astype(np.float32)

    # 3. DSO region
    device_geo = (
        df_train_ref[["deviceId", "latitude", "longitude"]]
        .drop_duplicates("deviceId")
    )
    device_geo["dso_region"] = device_geo.apply(
        lambda r: assign_dso(r["latitude"], r["longitude"]), axis=1
    )
    device_geo["nearest_centroid_km"] = device_geo.apply(
        lambda r: haversine_km(r["latitude"], r["longitude"],
                               *DSO_CENTROIDS[r["dso_region"]]), axis=1
    )
    df = df.merge(device_geo[["deviceId", "dso_region", "nearest_centroid_km"]],
                  on="deviceId", how="left")

    # 4. Norma klimatyczna IMGW per DSO per miesiąc
    df["imgw_t_mean_C"] = df.apply(
        lambda r: DSO_CLIMATE_NORMS.get(r["dso_region"], DSO_CLIMATE_NORMS["PGE"])[r["month"]],
        axis=1
    ).astype(np.float32)
    df["imgw_t_min_C"] = (df["imgw_t_mean_C"] - 5.0).astype(np.float32)

    # 5. Długość dnia (astronomiczna — dobrze ekstrapoluje na lato!)
    day_of_year_mid = df["day_of_year"]
    declination = 23.45 * np.sin(2 * np.pi * (284 + day_of_year_mid) / 365)
    lat_rad  = np.radians(df["latitude"].fillna(52.0))
    decl_rad = np.radians(declination)
    cos_ha   = (-np.tan(lat_rad) * np.tan(decl_rad)).clip(-1, 1)
    hour_angle = np.degrees(np.arccos(cos_ha))
    df["daylight_hours"] = (2 * hour_angle / 15).astype(np.float32)
    df["daylight_norm"]  = ((df["daylight_hours"] - 8) / 8).astype(np.float32)

    # Czy aktualnie jest dzień (przybliżenie)
    sunrise = (12 - df["daylight_hours"] / 2).clip(4, 10)
    sunset  = (12 + df["daylight_hours"] / 2).clip(14, 22)
    df["is_daylight"] = (
        (df["hour_of_day"] >= sunrise) & (df["hour_of_day"] <= sunset)
    ).astype(np.int8)

    # 6. Normy geo
    lat_min = df_train_ref["latitude"].min()
    lat_max = df_train_ref["latitude"].max()
    lon_min = df_train_ref["longitude"].min()
    lon_max = df_train_ref["longitude"].max()
    df["lat_norm"] = ((df["latitude"] - lat_min) / (lat_max - lat_min + 1e-9)).astype(np.float32)
    df["lon_norm"] = ((df["longitude"] - lon_min) / (lon_max - lon_min + 1e-9)).astype(np.float32)

    # 7. Statystyki per urządzenie z TRAINU (nie ma leakage dla val/test)
    dev_stats = (
        df_train_ref
        .groupby("deviceId")["x2_mean"]
        .agg(
            device_x2_mean_train   = "mean",
            device_x2_std_train    = "std",
            device_x2_p90_train    = lambda s: s.quantile(0.90),
            device_frac_high_train = lambda s: (s > 0.7).mean(),
        )
        .reset_index()
    )

    # Zachowanie w ciepłych vs zimnych godzinach
    warm_h = df_train_ref[df_train_ref["month"].isin([3, 4])]
    cold_h = df_train_ref[df_train_ref["month"].isin([1, 2, 11, 12])]
    dev_warm = warm_h.groupby("deviceId")["x2_mean"].mean().rename("device_x2_warm")
    dev_cold = cold_h.groupby("deviceId")["x2_mean"].mean().rename("device_x2_cold")

    dev_stats = dev_stats.merge(dev_warm, on="deviceId", how="left")
    dev_stats = dev_stats.merge(dev_cold, on="deviceId", how="left")

    global_warm = warm_h["x2_mean"].mean()
    global_cold = cold_h["x2_mean"].mean()
    dev_stats["device_x2_warm"] = dev_stats["device_x2_warm"].fillna(global_warm)
    dev_stats["device_x2_cold"] = dev_stats["device_x2_cold"].fillna(global_cold)
    dev_stats["device_warm_cold_ratio"] = (
        dev_stats["device_x2_warm"] / (dev_stats["device_x2_cold"] + 1e-6)
    ).astype(np.float32)

    # Liniowa krzywa temperaturowa per urządzenie (kluczowa dla ekstrapolacji!)
    dev_slopes = []
    for dev_id, grp in df_train_ref.groupby("deviceId"):
        if "t1_mean" in grp.columns:
            x = grp["t1_mean"].values
            y = grp["x2_mean"].values
            mask = ~(np.isnan(x) | np.isnan(y))
            if mask.sum() >= 3 and x[mask].std() > 1e-4:
                slope = np.cov(x[mask], y[mask])[0,1] / (np.var(x[mask]) + 1e-9)
                intercept = y[mask].mean() - slope * x[mask].mean()
            else:
                slope, intercept = 0.0, y[mask].mean() if mask.sum() > 0 else 0.1
        else:
            slope, intercept = 0.0, grp["x2_mean"].mean()
        dev_slopes.append({"deviceId": dev_id,
                           "dev_t1_slope":     float(slope),
                           "dev_t1_intercept": float(intercept)})

    dev_slopes_df = pd.DataFrame(dev_slopes)
    dev_stats = dev_stats.merge(dev_slopes_df, on="deviceId", how="left")

    df = df.merge(dev_stats, on="deviceId", how="left")

    # Przewidywana x2 z modelu liniowego przy danej temp zewnętrznej
    if "t1_mean" in df.columns:
        df["dev_linear_pred"] = (
            df["dev_t1_slope"] * df["t1_mean"] + df["dev_t1_intercept"]
        ).clip(0, 1).astype(np.float32)

    # 8. DSO target encoding (z trainu)
    # dso_means liczymy przez złączenie device_geo z df_train_ref
    tr_with_dso = df_train_ref.merge(
        device_geo[["deviceId", "dso_region"]], on="deviceId", how="left"
    )
    dso_means = tr_with_dso.groupby("dso_region")["x2_mean"].mean()
    global_x2_mean = df_train_ref["x2_mean"].mean()

    df["dso_x2_enc"] = df["dso_region"].map(dso_means).fillna(
        global_x2_mean
    ).astype(np.float32)

    return df


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Oblicz DSO raz z trainu, potem zastosuj feature engineering
# ─────────────────────────────────────────────────────────────────────────────

# Oblicz device_geo i dso_means RAZ z trainu
device_geo = (
    df_tr[["deviceId", "latitude", "longitude"]]
    .drop_duplicates("deviceId")
    .copy()
)
device_geo["dso_region"] = device_geo.apply(
    lambda r: assign_dso(r["latitude"], r["longitude"]), axis=1
)
device_geo["nearest_centroid_km"] = device_geo.apply(
    lambda r: haversine_km(r["latitude"], r["longitude"],
                           *DSO_CENTROIDS[r["dso_region"]]), axis=1
)

# Dołącz dso_region do trainu żeby można było liczyć dso_means
df_tr_with_dso = df_tr.merge(
    device_geo[["deviceId", "dso_region"]], on="deviceId", how="left"
)
dso_means = df_tr_with_dso.groupby("dso_region")["x2_mean"].mean().to_dict()
global_x2_mean = df_tr["x2_mean"].mean()

print("DSO means:", dso_means)

# Usuń sekcję 3 i 8 z funkcji — zastąp parametrami
def add_features_hourly(df, df_train_ref, device_geo, dso_means, global_x2_mean):
    df = df.copy()

    # 1. Cechy cykliczne czasu
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24).astype(np.float32)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24).astype(np.float32)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12).astype(np.float32)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12).astype(np.float32)
    df["dow_sin"]   = np.sin(2 * np.pi * df["day_of_week"] / 7).astype(np.float32)
    df["dow_cos"]   = np.cos(2 * np.pi * df["day_of_week"] / 7).astype(np.float32)
    df["doy_sin"]   = np.sin(2 * np.pi * df["day_of_year"] / 365).astype(np.float32)
    df["doy_cos"]   = np.cos(2 * np.pi * df["day_of_year"] / 365).astype(np.float32)
    df["is_peak_hour"] = df["hour_of_day"].isin([6,7,8,9,17,18,19,20,21]).astype(np.int8)
    df["is_night"]     = df["hour_of_day"].isin([0,1,2,3,4,5]).astype(np.int8)

    # 2. Delty temperatur
    if "t5_mean" in df.columns and "t4_mean" in df.columns:
        df["load_source_delta"] = (df["t5_mean"] - df["t4_mean"]).astype(np.float32)
    if "t6_mean" in df.columns and "t5_mean" in df.columns:
        df["load_hex_delta"]    = (df["t6_mean"] - df["t5_mean"]).astype(np.float32)
    if "t2_mean" in df.columns and "t1_mean" in df.columns:
        df["indoor_outdoor"]    = (df["t2_mean"] - df["t1_mean"]).astype(np.float32)
    if "t10_mean" in df.columns and "t11_mean" in df.columns:
        df["air_hex_delta"]     = (df["t10_mean"] - df["t11_mean"]).astype(np.float32)
    if "t8_mean" in df.columns and "t9_mean" in df.columns:
        df["cooling_ch_delta"]  = (df["t8_mean"] - df["t9_mean"]).astype(np.float32)
    if "t12_mean" in df.columns and "t13_mean" in df.columns:
        df["load_hex2_delta"]   = (df["t13_mean"] - df["t12_mean"]).astype(np.float32)

    # 3. DSO — z przekazanego device_geo
    df = df.merge(
        device_geo[["deviceId", "dso_region", "nearest_centroid_km"]],
        on="deviceId", how="left"
    )
    df["dso_x2_enc"] = df["dso_region"].map(dso_means).fillna(global_x2_mean).astype(np.float32)

    # 4. Norma klimatyczna IMGW
    df["imgw_t_mean_C"] = df.apply(
        lambda r: DSO_CLIMATE_NORMS.get(r["dso_region"], DSO_CLIMATE_NORMS["PGE"])[r["month"]],
        axis=1
    ).astype(np.float32)
    df["imgw_t_min_C"] = (df["imgw_t_mean_C"] - 5.0).astype(np.float32)

    # 5. Długość dnia
    day_of_year_mid = df["day_of_year"]
    declination = 23.45 * np.sin(2 * np.pi * (284 + day_of_year_mid) / 365)
    lat_rad  = np.radians(df["latitude"].fillna(52.0))
    decl_rad = np.radians(declination)
    cos_ha   = (-np.tan(lat_rad) * np.tan(decl_rad)).clip(-1, 1)
    hour_angle = np.degrees(np.arccos(cos_ha))
    df["daylight_hours"] = (2 * hour_angle / 15).astype(np.float32)
    df["daylight_norm"]  = ((df["daylight_hours"] - 8) / 8).astype(np.float32)
    sunrise = (12 - df["daylight_hours"] / 2).clip(4, 10)
    sunset  = (12 + df["daylight_hours"] / 2).clip(14, 22)
    df["is_daylight"] = (
        (df["hour_of_day"] >= sunrise) & (df["hour_of_day"] <= sunset)
    ).astype(np.int8)

    # 6. Geo normalizacja
    lat_min = df_train_ref["latitude"].min()
    lat_max = df_train_ref["latitude"].max()
    lon_min = df_train_ref["longitude"].min()
    lon_max = df_train_ref["longitude"].max()
    df["lat_norm"] = ((df["latitude"] - lat_min) / (lat_max - lat_min + 1e-9)).astype(np.float32)
    df["lon_norm"] = ((df["longitude"] - lon_min) / (lon_max - lon_min + 1e-9)).astype(np.float32)

    # 7. Statystyki per urządzenie z trainu
    dev_stats = (
        df_train_ref.groupby("deviceId")["x2_mean"]
        .agg(
            device_x2_mean_train   = "mean",
            device_x2_std_train    = "std",
            device_x2_p90_train    = lambda s: s.quantile(0.90),
            device_frac_high_train = lambda s: (s > 0.7).mean(),
        )
        .reset_index()
    )
    warm_h = df_train_ref[df_train_ref["month"].isin([3, 4])]
    cold_h = df_train_ref[df_train_ref["month"].isin([1, 2, 11, 12])]
    dev_warm = warm_h.groupby("deviceId")["x2_mean"].mean().rename("device_x2_warm")
    dev_cold = cold_h.groupby("deviceId")["x2_mean"].mean().rename("device_x2_cold")
    dev_stats = dev_stats.merge(dev_warm, on="deviceId", how="left")
    dev_stats = dev_stats.merge(dev_cold, on="deviceId", how="left")
    dev_stats["device_x2_warm"] = dev_stats["device_x2_warm"].fillna(warm_h["x2_mean"].mean())
    dev_stats["device_x2_cold"] = dev_stats["device_x2_cold"].fillna(cold_h["x2_mean"].mean())
    dev_stats["device_warm_cold_ratio"] = (
        dev_stats["device_x2_warm"] / (dev_stats["device_x2_cold"] + 1e-6)
    ).astype(np.float32)

    # Liniowa krzywa temperaturowa per urządzenie
    dev_slopes = []
    for dev_id, grp in df_train_ref.groupby("deviceId"):
        if "t1_mean" in grp.columns:
            x = grp["t1_mean"].values
            y = grp["x2_mean"].values
            mask = ~(np.isnan(x) | np.isnan(y))
            if mask.sum() >= 3 and x[mask].std() > 1e-4:
                slope = np.cov(x[mask], y[mask])[0,1] / (np.var(x[mask]) + 1e-9)
                intercept = y[mask].mean() - slope * x[mask].mean()
            else:
                slope, intercept = 0.0, y[mask].mean() if mask.sum() > 0 else 0.1
        else:
            slope, intercept = 0.0, grp["x2_mean"].mean()
        dev_slopes.append({"deviceId": dev_id,
                           "dev_t1_slope": float(slope),
                           "dev_t1_intercept": float(intercept)})

    dev_slopes_df = pd.DataFrame(dev_slopes)
    dev_stats = dev_stats.merge(dev_slopes_df, on="deviceId", how="left")
    df = df.merge(dev_stats, on="deviceId", how="left")

    if "t1_mean" in df.columns:
        df["dev_linear_pred"] = (
            df["dev_t1_slope"] * df["t1_mean"] + df["dev_t1_intercept"]
        ).clip(0, 1).astype(np.float32)

    return df


# Zastosuj na wszystkich zbiorach
print("Dodawanie cech do TRAIN...")
df_tr = add_features_hourly(df_tr, df_tr, device_geo, dso_means, global_x2_mean)

print("Dodawanie cech do VAL...")
df_vl = add_features_hourly(df_vl, df_tr, device_geo, dso_means, global_x2_mean)

print("Dodawanie cech do TEST...")
df_ts = add_features_hourly(df_ts, df_tr, device_geo, dso_means, global_x2_mean)

print(f"\nTrain: {df_tr.shape} | Val: {df_vl.shape} | Test: {df_ts.shape}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Zapis do Snowflake + eksport CSV
# ─────────────────────────────────────────────────────────────────────────────

# Zapisz do tabel Snowflake
session.write_pandas(df_tr, "HOURLY_TRAIN_FEATURES", auto_create_table=True, overwrite=True)
session.write_pandas(df_vl, "HOURLY_VAL_FEATURES",   auto_create_table=True, overwrite=True)
session.write_pandas(df_ts, "HOURLY_TEST_FEATURES",  auto_create_table=True, overwrite=True)

print("✓ Zapisano tabele:")
print(f"  HOURLY_TRAIN_FEATURES : {len(df_tr):,} wierszy")
print(f"  HOURLY_VAL_FEATURES   : {len(df_vl):,} wierszy")
print(f"  HOURLY_TEST_FEATURES  : {len(df_ts):,} wierszy")
print()
print("Pobierz CSV przez: Snowsight → Data → tabela → ⋮ → Download")

In [ ]:
SELECT 
    "deviceId",
    TO_TIMESTAMP("hour" / 1000000000) AS hour,
    "x1_mean", "x1_max", "x2_mean",
    "t1_mean", "t2_mean", "t3_mean", "t4_mean", "t5_mean", "t6_mean", "t7_mean",
    "t8_mean", "t9_mean", "t10_mean", "t11_mean", "t12_mean", "t13_mean",
    "latitude", "longitude",
    "year", "month", "day", "hour_of_day", "day_of_week", "day_of_year",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "doy_sin", "doy_cos", "is_peak_hour", "is_night",
    "load_source_delta", "load_hex_delta", "indoor_outdoor", "air_hex_delta",
    "cooling_ch_delta", "load_hex2_delta",
    "dso_region", "nearest_centroid_km", "dso_x2_enc",
    "imgw_t_mean_C", "imgw_t_min_C",
    "daylight_hours", "daylight_norm", "is_daylight",
    "lat_norm", "lon_norm",
    "device_x2_mean_train", "device_x2_std_train", "device_x2_p90_train",
    "device_frac_high_train", "device_x2_warm", "device_x2_cold",
    "device_warm_cold_ratio", "dev_t1_slope", "dev_t1_intercept", "dev_linear_pred"
FROM SNOWFLAKE_LEARNING_DB.PUBLIC.HOURLY_TRAIN_FEATURES; --tu zmieniamy zrodlo danych